# 🛒 Customer Intent Classification using XGBoost
---

## 📌 Problem Statement

When a customer visits a website or speaks to a sales agent, they leave behind clues — the words they use, how long they browse, how many times they've visited before. The big question a business always wants to answer is: **Is this person actually going to buy, or are they just looking around?** This mini-project tackles exactly that. We have a dataset of 520 customer interactions, where each row contains what a customer said (their "utterance"), how long their session lasted, how many times they had interacted before, and the follow-up status — and we want to automatically predict their **intent level**, ranging from "Casually Browsing" all the way up to "Strongly Interested". This is a 5-class classification problem, meaning the model must pick one of five possible intent categories for every customer. We use a combination of text analysis (reading what the customer actually said) and numerical signals (like session duration) to make that prediction. The model we use is called XGBoost — a powerful and fast machine learning algorithm that is widely used in industry for exactly these kinds of problems. By building this classifier, a business could automatically prioritize leads, trigger the right follow-up at the right time, and stop wasting resources on people who are nowhere near ready to buy.

---
##  Literature Survey

**On Intent Detection and Customer Behaviour Modelling**

Understanding what a customer wants — called "intent detection" — has been a key research area in both natural language processing and marketing analytics. Early work focused on rule-based systems where experts manually wrote rules like "if the customer says 'buy' or 'purchase', classify as high intent." These systems were brittle and didn't scale well. A more robust direction emerged with the use of supervised machine learning, where models are trained on labelled examples of customer conversations. Studies in e-commerce (e.g., research on session-based recommendation systems) showed that combining text features with behavioural signals — like how long someone spent on a page or how many items they viewed — significantly improved intent prediction accuracy compared to using text alone. This hybrid approach is exactly what this project adopts.

**On TF-IDF for Text Representation**

Before a machine learning model can process raw text, that text must be converted into numbers. One of the most widely studied and reliable techniques for this is TF-IDF (Term Frequency-Inverse Document Frequency), first formalised by Salton and McGill in the 1980s and still heavily used today. The idea is straightforward: words that appear frequently in one document but rarely across all documents are probably important and meaningful for that document. Research in text classification has consistently shown that TF-IDF is a strong baseline, often outperforming more complex embeddings on short, structured texts like customer utterances. For a task like ours, where customer messages are short and repetitive in vocabulary, TF-IDF is a sensible and proven choice.

**On XGBoost for Classification Tasks**

XGBoost (Extreme Gradient Boosting) was introduced by Chen and Guestrin in their 2016 paper "XGBoost: A Scalable Tree Boosting System" and has since become one of the most widely adopted machine learning algorithms in structured data tasks. It consistently won Kaggle competitions and real-world industry benchmarks in the years following its release. The key insight behind XGBoost is the concept of *gradient boosting* — building many small decision trees one after another, where each new tree specifically focuses on correcting the mistakes of the previous ones. Research has shown that XGBoost performs particularly well on tabular data (rows and columns), handles missing values gracefully, and is computationally efficient even on large datasets. For a mixed-feature problem like ours — where text vectors are combined with numerical columns — XGBoost is a well-justified choice backed by extensive empirical evidence.

---
##  Methodology and Results

**Data Preparation and Feature Engineering**

The dataset contains 520 rows of customer interaction data with 16 columns. From these, we selected four input features: the customer's spoken/written utterance (text), session duration in minutes (number), number of prior interactions (number), and the follow-up status (a category like "Pending" or "Yes - Closed"). The target variable is `Intent_Label`, which has 5 categories: Casually Browsing, Not Interested, Neutral, Interested, and Strongly Interested. The text column was converted to numbers using TF-IDF with a vocabulary of 1000 words, producing 1000 numeric columns per customer. The categorical follow-up status was converted to a number using Label Encoding (e.g., "Pending" → 1, "Yes - Closed" → 3). All features were combined into a single input matrix using NumPy's `hstack`, resulting in 1003 features per row (1000 from text + 3 numerical). The data was then split 80/20 into training and test sets.

**Model Architecture: How XGBoost Works**

XGBoost works by building decision trees one at a time in a sequential, error-correcting manner. A decision tree is like a flowchart — at each node, it asks a yes/no question (e.g., "Did the session last more than 15 minutes?") and branches accordingly until it reaches a prediction at the leaf. A single tree is often weak, but XGBoost builds 100 trees (`n_estimators=100`), and each new tree tries to fix the errors the previous trees made. The `learning_rate=0.1` controls how aggressively each new tree corrects errors — a smaller number means slower, more careful learning. The `max_depth=5` limits how many questions each tree can ask, preventing it from over-memorising the training data. The `eval_metric='mlogloss'` tells XGBoost to measure error using multi-class log loss, which is appropriate for a 5-class prediction problem. Since this is multi-class classification, XGBoost internally trains one set of trees per class and uses a softmax function to output probabilities.

**Results and Interpretation**

The model achieved a remarkable **98.08% accuracy** on the test set (104 samples). The classification report shows near-perfect precision and recall across all five intent categories, with F1-scores of 0.97–1.00. This is an excellent result, though it is worth noting that the high accuracy is partly a reflection of the dataset being relatively small (520 rows) and the intent labels being well-separated (customers saying "I want to buy this now!" vs. "I'm just browsing" are linguistically very different). The live prediction function at the end of the notebook demonstrates the model in action: when given the input "OMG I WANT TO BUY THIS", along with a 23-minute session, 8 prior interactions, and a scheduled follow-up, the model correctly predicted **Strongly Interested**. In production, such a system could be embedded into a CRM pipeline to automatically score leads in real time.

---
##  System Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                        RAW INPUT DATA                               │
│  Customer Utterance (text) + Session Duration + Prior Interactions  │
│                       + Follow-Up Status                            │
└───────────────────┬─────────────────────────┬───────────────────────┘
                    │                         │
                    ▼                         ▼
        ┌─────────────────────┐   ┌─────────────────────────┐
        │  TF-IDF Vectorizer  │   │    Label Encoder        │
        │  (text → 1000       │   │  (Follow-Up category    │
        │   numeric columns)  │   │   → single number)      │
        └──────────┬──────────┘   └────────────┬────────────┘
                   │                           │
                   └──────────┬────────────────┘
                              ▼
               ┌──────────────────────────────┐
               │   Combined Feature Matrix    │
               │  (1003 columns per customer) │
               └──────────────┬───────────────┘
                              │
                    ┌─────────▼────────┐
                    │   80% Training   │
                    │   20% Testing    │
                    └─────────┬────────┘
                              │
                              ▼
          ┌────────────────────────────────────────┐
          │         XGBoost Classifier             │
          │  100 trees | max_depth=5 | lr=0.1      │
          │  Tree 1 → Tree 2 → ... → Tree 100      │
          │  (each fixes errors of the previous)   │
          └──────────────────┬─────────────────────┘
                             │
                             ▼
          ┌────────────────────────────────────────┐
          │           PREDICTED INTENT             │
          │  Casually Browsing / Not Interested /  │
          │  Neutral / Interested / Strongly       │
          │  Interested                            │
          └────────────────────────────────────────┘
                    Accuracy: 98.08%
```

---
##  How XGBoost Works — Plain English Explanation

**What is a Decision Tree?**
Imagine you are trying to guess if someone wants to buy something. You might ask: "Did they say 'buy' or 'purchase'?" → Yes → probably interested. "Did they spend more than 20 minutes on the site?" → Yes → even more likely. This chain of yes/no questions is exactly what a decision tree does. It keeps asking questions until it reaches a final answer (a "leaf" node).

**Why use many trees instead of one?**
A single tree can make mistakes. So instead of relying on one tree, XGBoost builds 100 trees, but with a twist — each new tree is built specifically to fix the mistakes of all the previous trees combined. This is called **boosting**. Think of it like a relay race: Tree 1 does its best, Tree 2 focuses on the cases Tree 1 got wrong, Tree 3 focuses on what Tree 2 still got wrong, and so on.

**What does "Gradient" mean in Gradient Boosting?**
"Gradient" is just a mathematical way of saying "which direction do we need to move to reduce errors?" — like how water flows downhill. XGBoost uses this mathematical direction to guide each new tree towards reducing the overall prediction errors.

**Why XGBoost specifically (and not just any boosting)?**
XGBoost adds several smart improvements on top of basic gradient boosting: it uses **regularisation** to prevent overfitting (memorising training data), it handles missing values automatically, it can use multiple CPU cores to train fast, and it uses a clever system to find the best split points efficiently. That is why it is called "Extreme" — it is an optimised, production-grade version.

**What is TF-IDF?**
TF-IDF converts text into numbers. For each word in a customer message, it calculates how important that word is. Words like "buy", "purchase", "now" get high scores because they appear a lot in buying-intent messages but not everywhere. Common words like "the", "a", "is" get low scores because they appear everywhere and carry no useful meaning.

---
##  Cell 1: Importing Libraries
**What this does:** We load all the tools (libraries) we need before starting.

- `pandas` — for loading and managing our spreadsheet-like data (rows and columns)
- `numpy` — for doing math on large arrays of numbers
- `train_test_split` — splits our data into a training portion (what the model learns from) and a test portion (what we use to check if it learned correctly)
- `LabelEncoder` — converts text categories like "Pending" into numbers like 1, 2, 3 because machines only understand numbers
- `TfidfVectorizer` — converts raw customer text (utterances) into a matrix of numbers using the TF-IDF technique
- `accuracy_score`, `classification_report` — tools to measure how well the model performed
- `XGBClassifier` — the main XGBoost model we will train

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

---
##  Cell 2: Loading the Dataset
**What this does:** We read the Excel file into a pandas DataFrame — think of it as opening a spreadsheet in Python.

The dataset has **520 rows** and **16 columns**, including:
- `Customer_Utterance` — what the customer actually said (e.g., *"I'm ready to place the order"*)
- `Intent_Label` — the target we want to predict, with 5 possible values: *Casually Browsing, Not Interested, Neutral, Interested, Strongly Interested*
- `Session_Duration_Min` — how many minutes the customer spent in the session
- `Prior_Interactions` — how many times they have interacted before
- `Follow_Up_Status` — what happened next (e.g., "Pending", "Yes - Closed")
- And other columns like Channel, Device, Country, Age Group, etc.

`df.head()` just shows the first 5 rows so we can visually verify the data loaded correctly.

In [ ]:
df = pd.read_excel("customer_intent_classification_dataset.xlsx")
df.head()

---
##  Cell 3: Selecting Relevant Columns and Cleaning Data
**What this does:** We narrow down the dataset to only the columns we actually need for our model, then remove any rows with missing values.

**Why only these 5 columns?**
We keep:
1. `Customer_Utterance` — the customer's words (this is our main text signal)
2. `Session_Duration_Min` — how long they stayed (longer = more interested, usually)
3. `Prior_Interactions` — experience level / engagement history
4. `Follow_Up_Status` — whether a follow-up was scheduled or closed (a strong signal of intent)
5. `Intent_Label` — the answer we are trying to predict

Columns like Country, Device, Product_Category are dropped — not because they are useless, but to keep this model focused and simple.

`df.dropna(inplace=True)` removes any rows that have a blank/missing value. Models cannot handle empty cells, so this is necessary. Note: the `CTA_Clicked` column had many missing values (NaN), which is one reason it was excluded — and why after dropping NAs, we still have 520 rows (meaning only rows with all 5 chosen columns filled were kept).

In [ ]:
df = df[['Customer_Utterance',
         'Session_Duration_Min',
         'Prior_Interactions',
         'Follow_Up_Status',
         'Intent_Label']]

df.dropna(inplace=True)
df.head()

---
##  Cell 4: Encoding Categorical Columns
**What this does:** Machines only understand numbers. This step converts text categories into numbers.

**Label Encoding** assigns a unique integer to each unique category:

- `Follow_Up_Status`: "No" → 0, "Pending" → 1, "Unsubscribed" → 2, "Yes - Closed" → 3, "Yes - Follow-up Scheduled" → 4
- `Intent_Label` (our target): "Casually Browsing" → 0, "Interested" → 1, "Not Interested" → 2, "Neutral" → 3, "Strongly Interested" → 4 *(alphabetical order)*

We use **two separate LabelEncoders** (`le_follow` and `le_target`) so that we can later reverse the transformation on predictions — i.e., convert the predicted number back to the original label name (e.g., 4 → "Strongly Interested").

In [ ]:
le_follow = LabelEncoder()
df['Follow_Up_Status'] = le_follow.fit_transform(df['Follow_Up_Status'])

le_target = LabelEncoder()
df['Intent_Label'] = le_target.fit_transform(df['Intent_Label'])

---
##  Cell 5: Converting Customer Text to Numbers (TF-IDF)
**What this does:** Takes the `Customer_Utterance` column (raw text) and converts it into a big table of numbers that the model can read.

**How TF-IDF works, step by step:**
1. It looks at all 520 customer messages and builds a **vocabulary** of the 1000 most informative words (`max_features=1000`)
2. For each customer message, it creates a row of 1000 numbers — one number per word in the vocabulary
3. Each number reflects how important that word is in *this specific message* compared to *all messages*
4. Words like "buy", "order", "purchase" will have high values in messages from customers with high intent
5. Words like "maybe", "later", "think" will have high values in low-intent messages

`.toarray()` converts the result from a compressed sparse format into a regular 2D NumPy array (520 rows × 1000 columns).

After this step, `X_text` is a matrix of shape **(520, 1000)**.

In [ ]:
tfidf = TfidfVectorizer(max_features=1000)

X_text = tfidf.fit_transform(df['Customer_Utterance']).toarray()

---
##  Cell 6: Combining All Features and Defining Target
**What this does:** Brings together the text features and the numerical features into one unified input matrix, and separates out the target (what we want to predict).

- `X_other` — a (520, 3) matrix with the 3 numerical columns: session duration, prior interactions, follow-up status (now encoded as numbers)
- `np.hstack(...)` — "horizontal stack" — joins the two matrices side by side: the 1000 TF-IDF columns + the 3 numerical columns → **a final (520, 1003) matrix**
- `y` — our target labels (0–4 representing the 5 intent categories)

Think of `X` as a big grid where each row is a customer and each of the 1003 columns is a clue the model can use to figure out their intent.

In [ ]:
X_other = df[['Session_Duration_Min', 'Prior_Interactions', 'Follow_Up_Status']].values

X = np.hstack((X_text, X_other))
y = df['Intent_Label']

---
##  Cell 7: Splitting into Training and Test Sets
**What this does:** Divides our data into two parts — one to teach the model, one to test it.

- `test_size=0.2` means 20% of the data (104 rows) is set aside for testing; the remaining 80% (416 rows) is used for training
- `random_state=42` is just a seed number to make the split reproducible — using the same seed means every run of this notebook will produce the same train/test split (so results are consistent)

**Why split at all?** If we tested the model on the same data it trained on, it would look like it performs perfectly — but that is cheating. The test set is data the model has *never seen before*, so it acts like a real-world simulation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

---
##  Cell 8: Training the XGBoost Model
**What this does:** Creates the XGBoost classifier and trains it on our training data.

**Parameter breakdown:**
- `n_estimators=100` — build 100 decision trees in sequence. Each tree learns from the mistakes of the previous ones
- `max_depth=5` — each individual tree can ask at most 5 yes/no questions before making a prediction. Shallower trees are less likely to memorise noise in the data
- `learning_rate=0.1` — each new tree contributes only 10% of its correction to the final model. A lower rate means more careful, gradual learning (reduces overfitting)
- `eval_metric='mlogloss'` — tells XGBoost how to measure errors during training. Multi-class log loss is the standard metric for problems with more than 2 categories — it penalises confident wrong predictions more than uncertain wrong predictions

`.fit(X_train, y_train)` is where the actual learning happens. XGBoost iterates through building 100 trees, each one correcting the residual errors of the combined previous trees.

**How a single tree makes a split:**
At each node, XGBoost tests every possible split on every feature (e.g., "session_duration > 15?", "word 'buy' TF-IDF > 0.3?") and picks the split that reduces error the most. This is measured by a score called **gain**. The tree keeps splitting until it reaches `max_depth` or there is nothing useful left to split.

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

---
##  Cell 9: Evaluating the Model
**What this does:** Uses the trained model to predict on the test set (104 unseen rows) and measures how well it did.

**Output interpretation:**
- `Accuracy: 0.9807...` → the model got **98.08%** of test predictions correct (101 out of 104)

**Classification Report columns explained:**
- `Precision` — of all the times the model predicted a certain class, how often was it actually right? (e.g., precision of 1.00 for class 2 means every time it said "class 2", it was correct)
- `Recall` — of all the actual examples of a class, how many did the model catch? (e.g., recall of 0.93 for class 0 means it found 93% of the class 0 cases)
- `F1-Score` — the harmonic mean of precision and recall. A single balanced score. 1.0 is perfect.
- `Support` — how many actual examples of this class were in the test set

The model performs excellently across all 5 intent classes, with F1 scores ranging from **0.97 to 1.00**.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9807692307692307

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.96      1.00      0.98        22
           2       1.00      1.00      1.00        24
           3       0.95      1.00      0.98        21
           4       1.00      0.95      0.98        22

    accuracy                           0.98       104
   macro avg       0.98      0.98      0.98       104
weighted avg       0.98      0.98      0.98       104



---
##  Cell 10: Live Prediction Function
**What this does:** Defines a reusable function that takes new, unseen customer data as input and outputs the predicted intent label.

**Step-by-step breakdown of `predict_intent_full()`:**
1. **Collect inputs** — asks the user to type in: the customer utterance, session duration (in minutes), number of prior interactions, and the follow-up status (shown as a numbered menu)
2. **Transform the text** — uses the *same* TF-IDF object that was fit during training (`tfidf.transform(...)`) to convert the new utterance into the same 1000-number format. Note: we use `.transform()` (not `.fit_transform()`) here — because we must use the vocabulary learned during training, not re-learn from scratch
3. **Combine features** — stacks the TF-IDF vector with the 3 numerical inputs, exactly as was done during training
4. **Predict** — runs the combined 1003-feature input through all 100 trees and gets a predicted class number
5. **Decode** — uses `le_target.inverse_transform()` to convert the number back to the human-readable label (e.g., 4 → "Strongly Interested")

**Example run (from the notebook output):**
- Input: "OMG I WANT TO BUY THIS", 23 min session, 8 prior interactions, Follow-up Scheduled
- Output: 🔮 Predicted Intent: **Strongly Interested** ✅

In [ ]:
def predict_intent_full():
    # 1. Take user inputs
    utterance = input("Enter customer utterance: ")

    session_duration = float(input("Enter session duration (in minutes): "))

    prior_interactions = int(input("Enter number of prior interactions: "))

    # Show available follow-up options
    print("\nFollow-Up Status Options:")
    for i, label in enumerate(le_follow.classes_):
        print(f"{i}: {label}")

    follow_up_choice = int(input("Select follow-up status (enter number): "))

    follow_up = follow_up_choice

    # 2. Transform text
    text_vec = tfidf.transform([utterance]).toarray()

    # 3. Combine features
    other_features = np.array([[session_duration, prior_interactions, follow_up]])
    final_input = np.hstack((text_vec, other_features))

    # 4. Predict
    pred = model.predict(final_input)

    predicted_label = le_target.inverse_transform(pred)[0]

    print("\n🔮 Predicted Intent:", predicted_label)

---
## ▶ Cell 11: Running the Prediction
**What this does:** Calls the function defined above. This is the interactive demo of the model.

When run in Colab or a Jupyter notebook, it will prompt you for inputs one by one and then output the predicted intent label. This simulates how the model would work in a real application — imagine this being called from a CRM system or a chatbot backend whenever a sales rep finishes a call with a customer.

In [ ]:
predict_intent_full()

Enter customer utterance: OMG I WNAT TO BUY THIS
Enter session duration (in minutes): 23
Enter number of prior interactions: 8

Follow-Up Status Options:
0: No
1: Pending
2: Unsubscribed
3: Yes - Closed
4: Yes - Follow-up Scheduled
Select follow-up status (enter number): 4

🔮 Predicted Intent: Strongly Interested
